# Evaluation

The latent space of every run fetched by `scripts/download_results.sh`, side by side.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import pyarrow.parquet as pq
from plotly.subplots import make_subplots

from config.load import load_config
from config.paths import RESULTS_ROOT
from evaluation.metrics import (
    TOP_K,
    class_distances,
    class_separation,
    retrieval_metrics,
    silhouette_by_class,
    umap_projection,
)
from evaluation.results import RESULTS_FILE

In [ ]:
RUNS = sorted(path.parent.name for path in RESULTS_ROOT.glob(f"*/{RESULTS_FILE}"))
K = 10
SEED = load_config([]).dataset.seed

CLASS_COLORS = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
    "#8a8984",
]
CLASS_SYMBOLS = [
    "circle",
    "square",
    "diamond",
    "triangle-up",
    "triangle-down",
    "cross",
    "x",
    "star",
    "hexagon",
]
PRECISION_COLOR = "#e34948"
RECALL_COLOR = "#2a78d6"
BLUES = [
    [0.0, "#cde2fb"],
    [0.25, "#86b6ef"],
    [0.5, "#3987e5"],
    [0.75, "#1c5cab"],
    [1.0, "#0d366b"],
]
OUTLINE_COLOR = "#eb6834"
TEMPLATE = "plotly_white"
RUNS

In [ ]:
def neighbour_labels(distances: np.ndarray, labels: list[str]) -> np.ndarray:
    """Return the class of every other tile, nearest first, for each tile.

    Args:
        distances: The distance between every pair of tiles. (T, T)
        labels: The class each tile carries, in the same order.

    Returns:
        neighbours: The classes of each tile's neighbours, nearest first. (T, T-1)
    """
    held = np.asarray(labels)
    itself = np.eye(len(held), dtype=bool)
    return held[np.argsort(np.where(itself, np.inf, distances), axis=1)[:, :-1]]


def retrieval_by_tile(
    neighbours: np.ndarray, labels: list[str]
) -> tuple[np.ndarray, np.ndarray]:
    """Return every tile's precision and recall at each k of TOP_K.

    Args:
        neighbours: The classes of each tile's neighbours, nearest first. (T, T-1)
        labels: The class each tile carries, in the same order.

    Returns:
        precision: The share of each tile's k nearest tiles sharing its class. (K, T)
        recall: The share of its class those k nearest tiles reach. (K, T)
    """
    relevant = neighbours == np.asarray(labels)[:, None]
    total = np.maximum(relevant.sum(axis=1), 1)
    precision = np.stack([relevant[:, :k].mean(axis=1) for k in TOP_K])
    recall = np.stack([relevant[:, :k].sum(axis=1) / total for k in TOP_K])
    return precision, recall


def table(first: list[str], columns: dict[str, list[str]]) -> go.Figure:
    """A table of one row per name in the first column, then one column per entry."""
    figure = go.Figure(
        go.Table(
            header={
                "values": ["", *columns],
                "fill_color": "#f0efec",
                "align": "left",
                "height": 28,
            },
            cells={"values": [first, *columns.values()], "align": "left", "height": 24},
            columnwidth=[2, *[1] * len(columns)],
        )
    )
    figure.update_layout(
        template=TEMPLATE, height=48 + 24 * len(first), margin={"t": 10, "b": 10}
    )
    return figure


results = {}
for run in RUNS:
    held = pq.read_table(RESULTS_ROOT / run / RESULTS_FILE).to_pydict()
    distances = np.array(held["distances"])
    labels = held["label"]
    neighbours = neighbour_labels(distances, labels)
    precision, recall = retrieval_by_tile(neighbours, labels)
    logged = retrieval_metrics(distances, labels)
    assert all(
        np.isclose(precision[at].mean(), logged[f"precision@{k}"])
        and np.isclose(recall[at].mean(), logged[f"recall@{k}"])
        for at, k in enumerate(TOP_K)
    ), f"{run}: per-tile retrieval does not average to the logged one"
    classes, matrix = class_distances(distances, labels)
    results[run] = {
        "tiles": held["tile"],
        "labels": labels,
        "neighbours": neighbours,
        "precision": precision,
        "recall": recall,
        "logged": logged,
        "classes": classes,
        "matrix": matrix,
        "separation": class_separation(matrix),
        "closest": int((matrix.argmin(axis=1) == np.arange(len(matrix))).sum()),
        "silhouette": silhouette_by_class(distances, labels),
        "projection": umap_projection(distances, SEED),
    }
first = results[RUNS[0]]
for run in RUNS:
    assert sorted(results[run]["tiles"]) == sorted(first["tiles"]), (
        f"{run} holds other tiles than {RUNS[0]}"
    )
CLASSES = sorted(set(first["labels"]))
COUNTS = {name: first["labels"].count(name) for name in CLASSES}
TILES = len(first["tiles"])

## UMAP

In [ ]:
figure = make_subplots(rows=1, cols=len(RUNS), subplot_titles=RUNS)
for column, run in enumerate(RUNS, start=1):
    one = results[run]
    labels = np.asarray(one["labels"])
    tiles = np.asarray(one["tiles"])
    for at, name in enumerate(CLASSES):
        taken = labels == name
        figure.add_trace(
            go.Scatter(
                x=one["projection"][taken, 0],
                y=one["projection"][taken, 1],
                mode="markers",
                name=name,
                legendgroup=name,
                showlegend=column == 1,
                customdata=tiles[taken],
                hovertemplate=f"%{{customdata}}<br>{name}<extra></extra>",
                marker={
                    "size": 8,
                    "color": CLASS_COLORS[at],
                    "symbol": CLASS_SYMBOLS[at],
                    "line": {"width": 1, "color": "white"},
                },
            ),
            row=1,
            col=column,
        )
figure.update_xaxes(showticklabels=False)
figure.update_yaxes(showticklabels=False)
figure.update_layout(
    template=TEMPLATE, height=520, width=520 * len(RUNS) + 200, legend_title="class"
)
figure.show()

## Precision and recall

In [ ]:
rows = [f"{measure}@{k}" for measure in ("precision", "recall", "f1") for k in TOP_K]
chance_precision = sum(n * (n - 1) for n in COUNTS.values()) / (TILES * (TILES - 1))
chance = {
    **{f"precision@{k}": f"{chance_precision:.3f}" for k in TOP_K},
    **{f"recall@{k}": f"{k / (TILES - 1):.3f}" for k in TOP_K},
}
table(
    rows,
    {run: [f"{results[run]['logged'][row]:.3f}" for row in rows] for run in RUNS}
    | {"chance": [chance.get(row, "") for row in rows]},
).show()

In [ ]:
table(
    CLASSES,
    {
        "tiles": [str(COUNTS[name]) for name in CLASSES],
        "chance precision": [
            f"{(COUNTS[name] - 1) / (TILES - 1):.3f}" for name in CLASSES
        ],
        f"highest recall@{K}": [
            f"{min(K, COUNTS[name] - 1) / max(COUNTS[name] - 1, 1):.3f}"
            for name in CLASSES
        ],
    },
).show()

In [ ]:
order = sorted(
    range(len(first["tiles"])),
    key=lambda at: (first["labels"][at], first["tiles"][at]),
)
tiles = [first["tiles"][at] for at in order]
labels = [first["labels"][at] for at in order]
at_k = TOP_K.index(K)
bounds = [at for at in range(1, len(labels)) if labels[at] != labels[at - 1]]
starts = [0, *bounds]
ends = [*bounds, len(labels)]

figure = make_subplots(
    rows=len(RUNS),
    cols=1,
    shared_xaxes=True,
    subplot_titles=RUNS,
    vertical_spacing=0.08,
)
for row, run in enumerate(RUNS, start=1):
    one = results[run]
    where = {tile: at for at, tile in enumerate(one["tiles"])}
    held = [where[tile] for tile in tiles]
    for measure, color in (("precision", PRECISION_COLOR), ("recall", RECALL_COLOR)):
        figure.add_trace(
            go.Bar(
                x=list(range(len(tiles))),
                y=one[measure][at_k, held],
                name=f"{measure}@{K}",
                legendgroup=measure,
                showlegend=row == 1,
                marker={"color": color, "line": {"width": 0}},
                customdata=list(zip(tiles, labels, strict=True)),
                hovertemplate=(
                    f"%{{customdata[0]}}<br>%{{customdata[1]}}<br>{measure}@{K} "
                    "%{y:.2f}<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )
figure.update_xaxes(
    tickvals=[(start + end - 1) / 2 for start, end in zip(starts, ends, strict=True)],
    ticktext=[labels[start] for start in starts],
    tickangle=-30,
    range=[-0.5, len(tiles) - 0.5],
)
for start, end in list(zip(starts, ends, strict=True))[1::2]:
    figure.add_vrect(
        x0=start - 0.5,
        x1=end - 0.5,
        fillcolor="#f0efec",
        line_width=0,
        layer="below",
        row="all",
        col=1,
    )
figure.update_yaxes(range=[0, 1])
figure.update_layout(
    template=TEMPLATE,
    height=160 + 260 * len(RUNS),
    width=250 + 6 * len(tiles),
    barmode="group",
    bargap=0.15,
    bargroupgap=0.0,
)
figure.update_xaxes(title_text=f"query tile, by class ({len(tiles)})", row=len(RUNS))
figure.show()

## Distance within and between classes

In [ ]:
rows = ["within", "between", "separation"]
table(
    [*rows, "classes closest to themselves"],
    {
        run: [
            *[f"{results[run]['separation'][row]:.4f}" for row in rows],
            f"{results[run]['closest']}/{len(results[run]['classes'])}",
        ]
        for run in RUNS
    },
).show()

In [ ]:
for run in RUNS:
    one = results[run]
    figure = go.Figure(
        go.Heatmap(
            z=one["matrix"],
            x=one["classes"],
            y=one["classes"],
            colorscale=BLUES,
            reversescale=True,
            text=np.round(one["matrix"], 4),
            texttemplate="%{text}",
            hovertemplate="%{y} to %{x}: %{z:.4f}<extra></extra>",
            colorbar={"title": "mean distance"},
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[one["classes"][at] for at in one["matrix"].argmin(axis=1)],
            y=one["classes"],
            mode="markers",
            name="nearest class of the row",
            hoverinfo="skip",
            marker={
                "symbol": "square-open",
                "size": 40,
                "color": OUTLINE_COLOR,
                "line": {"width": 3},
            },
        )
    )
    figure.update_yaxes(autorange="reversed")
    figure.update_layout(
        template=TEMPLATE,
        title=(
            f"{run}: mean distance between two classes, darker is closer<br>"
            "<sup>outlined: the nearest class of each row</sup>"
        ),
        height=620,
        width=780,
    )
    figure.show()

## Nearest neighbours

In [ ]:
for run in RUNS:
    one = results[run]
    held = np.asarray(one["labels"])
    nearest = one["neighbours"][:, :K]
    shares = np.array(
        [
            [np.mean(nearest[held == row] == column) for column in CLASSES]
            for row in CLASSES
        ]
    )
    figure = go.Figure(
        go.Heatmap(
            z=shares,
            x=CLASSES,
            y=CLASSES,
            zmin=0,
            zmax=1,
            colorscale=BLUES,
            text=np.round(shares, 2),
            texttemplate="%{text}",
            hovertemplate=(
                "%{y} queries: %{z:.2f} of their neighbours are %{x}<extra></extra>"
            ),
            colorbar={"title": "share"},
        )
    )
    figure.update_xaxes(title_text=f"class of the {K} nearest tiles")
    figure.update_yaxes(title_text="query class", autorange="reversed")
    figure.update_layout(
        template=TEMPLATE,
        title=f"{run}: which classes a query's {K} nearest tiles belong to",
        height=620,
        width=780,
    )
    figure.show()

## Silhouette

In [ ]:
rows = ["silhouette", *[f"silhouette/{name}" for name in CLASSES]]
table(
    ["all", *CLASSES],
    {
        run: [
            f"{results[run]['silhouette'][row]:.3f}"
            if row in results[run]["silhouette"]
            else ""
            for row in rows
        ]
        for run in RUNS
    },
).show()